# VA-DCP — Screening Training A0/A1/A2

Sebelum pindah dari CPU, arsipkan A1/A2 dengan cell persistensi pada notebook setup. Setelah itu aktifkan T4 GPU dan jalankan notebook ini. Synthetic train dipulihkan dari Drive dan A0 diunduh ulang. Notebook melatih YOLO26n selama 10 epoch untuk screening seed 42; val/test selalu data nyata A0. Hasil dan checkpoint disimpan ke Drive dan dapat di-resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys
import torch

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
A0 = Path('/content/coffee-defect-roboflow')
A1 = Path('/content/coffee-A1-empirical-v10-full')
A2 = Path('/content/coffee-A2-vadcp-physics-v10-full')
AUDITS = Path('/content/drive/MyDrive/coffee-bean-detection/vadcp-setup-physics-v10/full')
ARCHIVES = Path('/content/drive/MyDrive/coffee-bean-detection/vadcp-full-v10')
OUTPUT = Path('/content/drive/MyDrive/coffee-bean-detection/vadcp-ablation-screen-v10')

assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > T4 GPU'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('GPU   :', torch.cuda.get_device_name(0))
print('ARCHIVE:', ARCHIVES)
print('OUTPUT:', OUTPUT)

In [ ]:
# Ambil runner terbaru dan instal pada kernel yang sama.
if REPO.is_dir():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
print('RUNNER SIAP')

In [ ]:
# Pulihkan synthetic train dari Drive setelah perpindahan CPU -> GPU.
from coffee_detector.archive_vadcp import restore_training_arm
restore_training_arm(ARCHIVES / 'A1_training.tar', A1)
restore_training_arm(ARCHIVES / 'A2_training.tar', A2)

# A0 lebih efisien diunduh ulang daripada diarsipkan ke Drive.
if not (A0 / 'data.yaml').is_file():
    from getpass import getpass
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'], check=True)
    from roboflow import Roboflow
    rf = Roboflow(api_key=getpass('Masukkan API key Roboflow: '))
    dataset = rf.workspace('adrians').project('coffee-defect-1eny4').version(11).download('yolov8', location=str(A0), overwrite=True)
    A0 = Path(dataset.location).resolve()
for name, root in {'A0': A0, 'A1': A1, 'A2': A2}.items():
    assert (root / 'data.yaml').is_file(), f'{name} belum siap: {root}'
for audit in ('source_dataset_audit.json', 'A1_vadcp_audit.json', 'A2_vadcp_audit.json'):
    assert (AUDITS / audit).is_file(), f'Audit hilang: {AUDITS / audit}'
print('A0/A1/A2 SIAP PADA RUNTIME GPU')

In [ ]:
# Screening terkontrol: 10 epoch, seed 42, sequential A0 -> A1 -> A2.
# Rerun cell ini aman: checkpoint last.pt akan di-resume dan run selesai akan di-skip.
command = [
    sys.executable, '-u', '-m', 'coffee_detector.run_vadcp_ablation',
    '--arm', f'A0={A0}',
    '--arm', f'A1={A1}',
    '--arm', f'A2={A2}',
    '--verified-audit', f'A0={AUDITS / "source_dataset_audit.json"}',
    '--verified-audit', f'A1={AUDITS / "A1_vadcp_audit.json"}',
    '--verified-audit', f'A2={AUDITS / "A2_vadcp_audit.json"}',
    '--config', f'A0={REPO / "configs/A0_yolo26n_screen.yaml"}',
    '--config', f'A1={REPO / "configs/A1_yolo26n_screen.yaml"}',
    '--config', f'A2={REPO / "configs/A2_yolo26n_screen.yaml"}',
    '--seeds', '42',
    '--output-root', str(OUTPUT),
    '--device', '0',
]
print('MULAI SCREENING A0/A1/A2. Progress Ultralytics akan tampil di bawah.', flush=True)
subprocess.run(command, check=True)

In [ ]:
# Ringkasan hanya tersedia setelah ketiga arm selesai.
import json
summary_path = OUTPUT / 'reports' / 'vadcp_ablation_summary.json'
assert summary_path.is_file(), f'Screening belum lengkap: {summary_path}'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('=== AGGREGATE ===')
print(json.dumps(summary['aggregate'], indent=2, ensure_ascii=False))
print('=== DELTA VS A0 ===')
print(json.dumps(summary['comparisons'], indent=2, ensure_ascii=False))
print('SAVED:', summary_path)